In [1]:
import json
import random
import base64
import os
import shutil
import subprocess
import dill
import nltk
import config
import itertools
import io
import math
import textwrap
import taxonomy
import time

from PIL import Image
from utils.DataHandlers import LatexDataHandler
from utils.ModelClient import ModelClient
from pathlib import Path
from collections import Counter
from pdf2image import convert_from_bytes

In [2]:
EDIT_TYPE = config.EDIT_TYPE
DATA_FOLDER_PATH = config.DATA_FOLDER_PATH
TASK_ID = 'identification_and_localization'
INPUT_TYPE = 'images'
COMPILATION_TYPE = config.COMPILATION_TYPE

In [3]:
def get_system_prompt():
    system_prompt = textwrap.dedent("""
        # Identity
        You are a peer-review assistant that identifies errors and weaknesses in scientific manuscripts.

        # Task
        You will be provided with a scientific manuscript. Your task is to identify and locate any errors, weaknesses, or inconsistencies between the claims made in the text and the values presented in the tables. You should pay particular attention to the types of errors and weaknesses described below.
                                    
        """[1:]
    )

    error_extension = textwrap.dedent("""
        # Error or Weakness
        This section contains a list of error or weakness types and their corresponding definitions.

        """[1:]
    )

    for error_category in taxonomy.TAXONOMY_DATA.keys():
        description = taxonomy.TAXONOMY_DATA[error_category]['description']
        error_name = taxonomy.TAXONOMY_DATA[error_category]['name']
        error_extension += f"## {error_name}\n{description}\n\n"

    req = textwrap.dedent("""
        # Requirements
        * You should be as specific as possible; try to provide the specific sentences or tables in the manuscript that are relevant to the identified error or weakness.
        * You should be as comprehensive as possible; identify all weaknesses or errors in the manuscript, even if the source of the issue is the same.
        * If more than one of the categories described above is relevant to an error or weakness, you should explain and attempt to localize it for each relevant category.
        * You should focus on identifying weaknesses and errors. Do not abstain from responding, do not include irrelevant information even if it may be helpful, and do not ask follow-up questions.
        """[1:]
    )

    return system_prompt + error_extension + req

In [4]:
SYSTEM_PROMPT = get_system_prompt()

MESSAGE_TEMPLATE_IMAGES = """Identify any inconsistencies between the text and tables in the provided manuscript. You are provided with images of the pages of the manuscript."""

MESSAGE_TEMPLATE_LATEX = """Identify any inconsistencies between the text and tables in the provided manuscript. You are provided with the latex source of the manuscript.

# Manuscript
{0}
"""

MESSAGE_TEMPLATE_IMAGES_AND_LATEX = """# Manuscript
{0}

Identify any inconsistencies between the text and tables in the provided manuscript. You are provided with the images of the pages of the manuscript and the latex source of the manuscript.
"""



In [5]:
# from prismm-bench code in llms-eval: https://github.com/EvolvingLMMs-Lab/lmms-eval/tree/main/lmms_eval/tasks/prismm_bench
def concat_images(images, column_num=3):
    if not images:
        return None

    if column_num == 1:
        total_height = images[0].height * len(images)
        total_width = images[0].width
    else:
        rows = math.ceil(len(images) / column_num)
        total_height = images[0].height * rows
        total_width = images[0].width * column_num

    concatenated_image = Image.new("RGB", (total_width, total_height), "white")

    x_offset, y_offset = 0, 0
    for cnt, image in enumerate(images):
        concatenated_image.paste(image, (x_offset, y_offset))
        x_offset += image.width
        if (cnt + 1) % column_num == 0:
            y_offset += image.height
            x_offset = 0

    return concatenated_image

def get_pdf_bytes(doc):
    pdf_data = doc.get("pdf")
    if pdf_data is None:
        raise ValueError("PDF data not found in document")
    return pdf_data.stream.getvalue()

# modified from prismm-bench code in llms-eval: https://github.com/EvolvingLMMs-Lab/lmms-eval/tree/main/lmms_eval/tasks/prismm_bench
def paper_to_img_grid(paper_path):
    visuals = []
    # pdf_bytes = get_pdf_bytes(doc)
    
    with open(paper_path, 'rb') as pdf:
        pdf_bytes = pdf.read()

    if not pdf_bytes:
        return visuals

    try:
        images = convert_from_bytes(pdf_bytes, dpi=144, last_page=50)  # safeguard for very large documents
        max_images = 5
        column_num = 3

        if not images:
            return visuals

        pages_per_image = math.ceil(len(images) / max_images)

        for i in range(0, len(images), pages_per_image):
            batch_pages = images[i : i + pages_per_image]
            concatenated_image = concat_images(batch_pages, column_num)
            if concatenated_image:
                visuals.append(concatenated_image.convert("RGB"))
            if len(visuals) >= max_images:
                break
    except Exception as e:
        print(e)
        pass

    return visuals

def prepare_image_req_data(paper_path):
    if f'{paper_path.stem}_visual_grid.dill' in os.listdir(paper_path.parent):
        with open(paper_path.parent / f'{paper_path.stem}_visual_grid.dill', 'rb') as fp:
            img_data = dill.load(fp)
    else:
        visuals = paper_to_img_grid(paper_path)

        img_data = []
        for vis in visuals:
            img_byte_arr = io.BytesIO()
            vis.save(img_byte_arr, format='PNG')
            img_byte_arr = img_byte_arr.getvalue()
            base64_img = base64.b64encode(img_byte_arr).decode('utf-8')
            img_data.append(base64_img)

        with open(paper_path.parent / f'{paper_path.stem}_visual_grid.dill', 'wb') as fp:
            dill.dump(img_data, fp)

    req_data = {
        'system_prompt': SYSTEM_PROMPT,
        'message': MESSAGE_TEMPLATE_IMAGES,
        'img_data': img_data,
    }

    return req_data

def prepare_latex_req_data(paper_path):
    with open(paper_path, 'r') as fp:
        latex = fp.read()    

    req_data = {
        'system_prompt': SYSTEM_PROMPT,
        'message': MESSAGE_TEMPLATE_LATEX.format(latex),
    }

    return req_data

def prepare_images_and_latex_req_data(paper_path_images, paper_path_latex):
    req_data = prepare_image_req_data(paper_path_images)
    with open(paper_path_latex, 'r') as fp:
        latex = fp.read()    

    req_data['message'] = MESSAGE_TEMPLATE_IMAGES_AND_LATEX.format(latex)
    return req_data

def manuscript_iterator(project_data):
    for paper_id, error_id, data in project_data:
        meta_id = f'{paper_id}___{error_id}' #combined paper + error id
        for input_type in ['images', 'images_and_latex', 'latex']:
            meta_id_input = meta_id + '___' + input_type
            if input_type == 'images':
                paper_path = data['edited_pdf_path']
                req_data = prepare_image_req_data(paper_path)
            elif input_type == 'latex':
                paper_path = data['edited_latex_path']
                req_data = prepare_latex_req_data(paper_path)
            elif input_type == 'images_and_latex':
                paper_path_images = data['edited_pdf_path']
                paper_path_latex = data['edited_latex_path']
                req_data = prepare_images_and_latex_req_data(paper_path_images, paper_path_latex)
            
            req_data['reasoning_effort'] = "high"
            yield (meta_id_input, req_data)

        # if INPUT_TYPE == 'pdf':
        #     with open(paper_path, 'rb') as fp:
        #         data = fp.read()
        #     file_data = base64.b64encode(data).decode("utf-8")
        #     req_data = {
        #         'system_prompt': SYSTEM_PROMPT,
        #         'message': MESSAGE_TEMPLATE,
        #         'file_data': file_data,
        #     }
        # elif INPUT_TYPE == 'images':
        #     # Handle image input type
        #     req_data = prepare_image_req_data(paper_path)
        # elif INPUT_TYPE == 'images_and_latex':
        #     pass

        # file_data = base64.b64encode(data).decode("utf-8")
        # meta_id = paper_path.stem[5:] #combined paper + error id
        # req_data = {
        #     'system_prompt': SYSTEM_PROMPT,
        #     'message': MESSAGE_TEMPLATE,
        #     'file_data': file_data,
        # }
        # yield (meta_id, req_data)

In [6]:
# DETECTION_CLIENTS = []

# for dmi in config.DETECTION_MODELS_INFO:
#     DETECTION_CLIENTS.append(ModelClient(*dmi))

In [7]:
DETECTION_CLIENTS = [
    #ModelClient("openai", "gpt-5.4-nano-2026-03-17", "gpt_54_nano", "output/detect_gpt54_nano_workspace"),
    #ModelClient("openai", "gpt-5.4-mini-2026-03-17", "gpt_54_mini", "output/detect_gpt54_mini_workspace"),
    #ModelClient("openai", "gpt-5.4-2026-03-05", "gpt_54", "output/detect_gpt54_workspace"),
    #ModelClient("openrouter", "google/gemini-2.5-flash-lite", "gemini_25_flash", "output/detect_gemini_25_flash_workspace"),
    #ModelClient("gemini", "gemma-4-31b-it", "gemma_4_31b", "output/detect_gemma4_workspace"),
    #ModelClient("gemini", "gemini-2.5-flash", "gemini_25_flash", "output/detect_gemini_25_flash_workspace")
    #ModelClient("gemini", "gemini-2.5-flash-lite", "gemini_25_flash_lite", "output/detect_gemini_25_flash_lite_workspace")
    #ModelClient("gemini", "gemini-3.1-flash-lite", "gemini_31_flash_lite", "output/detect_gemini_31_flash_lite_workspace"),
    #----LOW REASONING----
    # ModelClient("openai", "gpt-5.4-nano-2026-03-17", "gpt_54_nano_low", "output/detect_gpt54_nano_workspace"),
    # ModelClient("openai", "gpt-5.4-mini-2026-03-17", "gpt_54_mini_low", "output/detect_gpt54_mini_workspace"),
    #----Open Router----
    ModelClient("openrouter", "qwen/qwen3-vl-30b-a3b-thinking", "qwen_3_vl_30b_a3b_thinking", "output/qwen_3_vl_30b_a3b_thinking"), #CONTEXT LENGTH 131K WHEN USING ALIBABA
    ModelClient("openrouter", "qwen/qwen3-vl-8b-thinking", "qwen_3_vl_8b_thinking", "output/qwen_3_vl_8b_thinking"), #CONTEXT LENGTH 131K WHEN USING ALIBABA
    ModelClient("openrouter", "qwen/qwen3-vl-8b-instruct", "qwen_3_vl_8b_instruct", "output/qwen_3_vl_8b_instruct"), #CONTEXT LENGTH 131K WHEN USING ALIBABA
    #ModelClient("openrouter", "qwen/qwen3-vl-30b-a3b-instruct", "qwen_3_vl_30b_a3b_instruct", "output/qwen_3_vl_30b_a3b_instruct"), #CONTEXT LENGTH 131K WHEN USING ALIBABA
    #ModelClient("openrouter", "qwen/qwen3-vl-32b-instruct", "qwen_3_vl_32b_instruct", "output/qwen_3_vl_32b_instruct"),
]

In [8]:
with open('runtime.txt', 'w') as fp:
    ts = time.time()
    fp.write(str(ts))

In [9]:
for model_client in DETECTION_CLIENTS:
    project_ids = []
    with open('./output/responses_edit_latex_gpt_54_compiled.dill', 'rb') as fp:
        compiled_data = dill.load(fp)

        project_data = []
        for project_id, error_id, compiled in compiled_data:
            if compiled['valid'] and compiled["edited_pdf_path"] is not None:
                project_data.append((project_id, error_id, compiled))
    
    # manuscript_file_paths = [project_data[2] for project_data in project_ids]
    cache_path = f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}_cache'
    count = 0
    for meta_id, res in model_client.iterative_query(manuscript_iterator(project_data), cache_file_path=cache_path, batch=False, retry_cached_empty_response=True):
        paper_id, error_id, input_type = meta_id.split('___')
        error_id = meta_id.split('___')[1]
        print(meta_id)
        res_dict = {
            'model_id': model_client.model_prefix_name,
            'task_id': TASK_ID,
            'input_type': input_type,
            'response_text': res
        }

        for compiled in compiled_data:
            if compiled[0] == paper_id and compiled[1] == error_id:
                if 'model_responses' in compiled[2]:
                    compiled[2]['model_responses'].append(res_dict)
                else:
                    compiled[2]['model_responses'] = [res_dict]
                break

    with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_client.model_prefix_name}.dill', 'wb') as fp:
        dill.dump(compiled_data, fp)

Loading from cache...
Loading cached responses from output/responses_identification_and_localization_images_qwen_3_vl_30b_a3b_thinking_cache.dill
2409.14664___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___images
[2409.14664___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___images_and_latex: empty response in cache, skipping load
2409.14664___bWV0aG9kb2xvZ3lfcmVzdWx0X2NvbmZsaWN0___latex
2409.14664___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=___images
2409.14664___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=___images_and_latex
2409.14664___c2VtYW50aWNfbWlzaW50ZXJwcmV0YXRpb24=___latex
2409.14664___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___images
[2409.14664___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___images_and_latex: empty response in cache, skipping load
2409.14664___c3Vib3B0aW1hbF9tZXRob2RvbG9neQ==___latex
[2409.14664___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==___images: empty response in cache, skipping load
2409.14664___Y29udHJhZGljdGlvbl93aXRoX2VzdGFibGlzaGVkX2tub3dsZWRnZQ==___images_and_latex
2409.14664___Y29u

In [25]:
# make sure all model responses are recorded
model_prefix = "qwen_3_vl_8b_instruct"
with open(f'{config.OUTPUT_FOLDER_PATH}responses_{TASK_ID}_{INPUT_TYPE}_{model_prefix}.dill', 'rb') as fp:
    responses = dill.load(fp)

print(f'Total responses: {len(responses)}')
count = 0
incomplete = 0
empty_response = 0
for mid, errid, res in responses:
    if res['valid'] and res['edited_pdf_path'] and 'model_responses' not in res:
        count += 1
    elif 'model_responses' in res:
        #one response for each modality type
        if len(res['model_responses']) < 3:
            incomplete += 1
        else:
            for mr in res['model_responses']:
                if mr['response_text'] is None:
                    empty_response += 1
                #empty response
                elif len(mr['response_text']) == 0:
                    empty_response += 1

print(f'Number of missing model responses: {count}') 
print(f'Number of incomplete model responses for manuscript/error pair: {incomplete}') 
print(f'Number of empty model responses for manuscript/error pair: {empty_response}') 

Total responses: 300
Number of missing model responses: 0
Number of incomplete model responses for manuscript/error pair: 0
Number of empty model responses for manuscript/error pair: 1
